In [5]:
from neo4j import GraphDatabase
import pandas as pd
from tqdm import tqdm  # 진행률 시각화

# 데이터 로딩
df = pd.read_csv("청년정책목록_전체.csv")

# Neo4j 연결 설정
uri = "bolt://localhost:7687"
user = "neo4j"
password = "neo4j1234"  # ← 비밀번호 변경

driver = GraphDatabase.driver(uri, auth=(user, password))

def create_nodes_and_relationships(tx, row):
    # 정책 노드
    tx.run("""
        MERGE (p:Policy {id: $plcyNo})
        SET p.name = $plcyNm, p.description = $plcyExplnCn
    """, plcyNo=row['plcyNo'], plcyNm=row['plcyNm'], plcyExplnCn=row['plcyExplnCn'])

    # 주관 기관
    if pd.notna(row['sprvsnInstCdNm']):
        tx.run("""
            MERGE (o:Organization {name: $org})
            MERGE (p:Policy {id: $plcyNo})-[:PROVIDED_BY]->(o)
        """, org=row['sprvsnInstCdNm'], plcyNo=row['plcyNo'])

    # 정책 대분류
    if pd.notna(row['lclsfNm']):
        major_categories = str(row['lclsfNm']).split(',')
        for major_category in major_categories:
            major_category = major_category.strip()
            # 대분류 노드 생성 및 정책과 연결
            tx.run("""
                MERGE (l:MajorCategory {name: $majorCategory})
                MERGE (p:Policy {id: $plcyNo})-[:BELONGS_TO]->(l)
            """, majorCategory=major_category, plcyNo=row['plcyNo'])

    # 정책 중분류 (쉼표 분리)
    if pd.notna(row['mclsfNm']):
        categories = str(row['mclsfNm']).split(',')
        for cat in categories:
            cat = cat.strip()
            tx.run("""
                MERGE (c:Category {name: $category})
                MERGE (p:Policy {id: $plcyNo})-[:BELONGS_TO]->(c)
            """, category=cat, plcyNo=row['plcyNo'])

    # 지원대상
    if pd.notna(row['sprtTrgtMinAge']) and pd.notna(row['sprtTrgtMaxAge']):
        age_range = f"{int(row['sprtTrgtMinAge'])}-{int(row['sprtTrgtMaxAge'])}"
        tx.run("""
            MERGE (a:SprtTrgt {range: $range})
            MERGE (p:Policy {id: $plcyNo})-[:TARGETS]->(a)
        """, range=age_range, plcyNo=row['plcyNo'])

    # 키워드
    if pd.notna(row['plcyKywdNm']):
        keywords = str(row['plcyKywdNm']).split(',')
        for kw in keywords:
            kw = kw.strip()
            tx.run("""
                MERGE (k:Keyword {name: $keyword})
                MERGE (p:Policy {id: $plcyNo})-[:HAS_KEYWORD]->(k)
            """, keyword=kw, plcyNo=row['plcyNo'])

    # 지역
    if pd.notna(row['rgtrHghrkInstCdNm']):
        tx.run("""
            MERGE (r:Region {name: $region})
            MERGE (p:Policy {id: $plcyNo})-[:AVAILABLE_IN]->(r)
        """, region=row['rgtrHghrkInstCdNm'], plcyNo=row['plcyNo'])

# 삽입 실행 (진행률 표시)
with driver.session() as session:
    for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc="Neo4j Insertion"):
        session.write_transaction(create_nodes_and_relationships, row)

driver.close()


Neo4j Insertion:   0%|          | 0/3458 [00:00<?, ?it/s]/tmp/ipykernel_26195/2490272129.py:78: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_nodes_and_relationships, row)
Neo4j Insertion: 100%|██████████| 3458/3458 [00:43<00:00, 78.97it/s] 


In [4]:
from neo4j import GraphDatabase

# Neo4j 연결 설정
uri = "bolt://localhost:7687"
user = "neo4j"
password = "neo4j1234"

driver = GraphDatabase.driver(uri, auth=(user, password))

def delete_all_data():
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        print("✅ 모든 노드와 관계가 삭제되었습니다.")

# 실행
if __name__ == "__main__":
    delete_all_data()


✅ 모든 노드와 관계가 삭제되었습니다.


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 60 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   plcyNo             3458 non-null   object 
 1   bscPlanCycl        3458 non-null   int64  
 2   bscPlanPlcyWayNo   3458 non-null   int64  
 3   bscPlanFcsAsmtNo   3458 non-null   int64  
 4   bscPlanAsmtNo      3458 non-null   int64  
 5   pvsnInstGroupCd    3458 non-null   int64  
 6   plcyPvsnMthdCd     623 non-null    float64
 7   plcyAprvSttsCd     3458 non-null   int64  
 8   plcyNm             3458 non-null   object 
 9   plcyKywdNm         956 non-null    object 
 10  plcyExplnCn        3458 non-null   object 
 11  lclsfNm            3421 non-null   object 
 12  mclsfNm            3421 non-null   object 
 13  plcySprtCn         3458 non-null   object 
 14  sprvsnInstCd       2464 non-null   object 
 15  sprvsnInstCdNm     3440 non-null   object 
 16  sprvsnInstPicNm    766 n

In [7]:
df['plcyPvsnMthdCd']

0       42013.0
1       42010.0
2       42005.0
3       42005.0
4       42002.0
         ...   
3453        NaN
3454        NaN
3455        NaN
3456        NaN
3457    42004.0
Name: plcyPvsnMthdCd, Length: 3458, dtype: float64